In [4]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score


In [5]:
df = pd.read_csv('heart.csv')

In [6]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [7]:
df.shape

(303, 14)

In [8]:
#need to predict they have heart disease or not

In [9]:
X = df.iloc[: , 0 : -1]
y = df.iloc[: , -1]

In [10]:
#it splits the data the data into training and test where x is the input and y is the output
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
print(X_train.shape)
print(X_test.shape)

(242, 13)
(61, 13)


In [12]:
rf = RandomForestClassifier()
gb = GradientBoostingClassifier()
svc = SVC()
lr = LogisticRegression()

In [13]:
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test , y_pred)

0.8688524590163934

In [14]:
svc.fit(X_train,y_train)
y_pred = svc.predict(X_test)
accuracy_score(y_test , y_pred)

0.7049180327868853

In [15]:
gb.fit(X_train,y_train)
y_pred = gb.predict(X_test)
accuracy_score(y_test , y_pred)

0.7704918032786885

In [16]:
lr.fit(X_train,y_train)
y_pred = lr.predict(X_test)
accuracy_score(y_test , y_pred)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8852459016393442

In [17]:
rf = RandomForestClassifier( max_samples = 0.75 , random_state = 42)
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test , y_pred)

0.9016393442622951

In [18]:
from sklearn.model_selection import cross_val_score

np.mean(
    cross_val_score(
        RandomForestClassifier(max_samples = 0.75),
        X,
        y,
        cv = 10,
        scoring = 'accuracy'
    )
)

np.float64(0.8348387096774192)

####Grid Search CV


In [19]:
n_estimators = [20,60,100,120]

max_features = [0.2,0.6,1.0]

max_depth = [2,8,None]

max_samples = [0.25,0.5,0.75,1.0]

In [20]:
param_grid = {
    'n_estimators' : n_estimators,
    'max_features' : max_features,
    'max_depth' : max_depth,
    'max_samples' : max_samples
}
print(param_grid)

{'n_estimators': [20, 60, 100, 120], 'max_features': [0.2, 0.6, 1.0], 'max_depth': [2, 8, None], 'max_samples': [0.25, 0.5, 0.75, 1.0]}


In [21]:
rf = RandomForestClassifier()

In [22]:
from sklearn.model_selection import GridSearchCV

rf_grid = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    cv = 5,
    n_jobs = -1,
    verbose = 2
)

In [23]:
rf_grid.fit(X_train,y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'max_depth': [2, 8, None],
                         'max_features': [0.2, 0.6, 1.0],
                         'max_samples': [0.25, 0.5, 0.75, 1.0],
                         'n_estimators': [20, 60, 100, 120]},
             verbose=2)

In [24]:
rf_grid.best_params_

{'max_depth': 8, 'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 20}

In [25]:
rf_grid.best_score_

np.float64(0.8345238095238097)

####Random Search CV
######generally used for large datasets not because of the accuracy but bcs it takes lesser time as comp to grid_search_cv (only for large)

In [26]:
n_estimators = [20,60,100,120]

max_features = [0.2,0.6,1.0]

max_depth = [2,8,None]

max_samples = [0.5 , 0.75 , 1.0]

bootstrap = [True , False]

min_samples_split = [2,5]

min_samples_leaf = [1,2]



In [27]:
param_grid = {
    'n_estimators' : n_estimators,
    'max_features' : max_features,
    'max_depth' : max_depth,
    'max_samples' : max_samples,
    'bootstrap' : bootstrap,
    'min_samples_split' : min_samples_split,
    'min_samples_leaf' : min_samples_leaf
}
print(param_grid)

{'n_estimators': [20, 60, 100, 120], 'max_features': [0.2, 0.6, 1.0], 'max_depth': [2, 8, None], 'max_samples': [0.5, 0.75, 1.0], 'bootstrap': [True, False], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}


In [28]:
from sklearn.model_selection import RandomizedSearchCV
rf_grid = RandomizedSearchCV(
    estimator = rf,
    param_distributions = param_grid,

    cv = 5,
    n_jobs = -1,
    verbose = 2
)

In [29]:
rf_grid.fit(X_train , y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_forest.py", line 431, in fit
    raise ValueError(
ValueError:

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [2, 8, None],
                                        'max_features': [0.2, 0.6, 1.0],
                                        'max_samples': [0.5, 0.75, 1.0],
                                        'min_samples_leaf': [1, 2],
                                        'min_samples_split': [2, 5],
                                        'n_estimators': [20, 60, 100, 120]},
                   verbose=2)

In [30]:
rf_grid.best_params_

{'n_estimators': 100,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_samples': 0.5,
 'max_features': 0.2,
 'max_depth': None,
 'bootstrap': True}